# 02 — Fixed FP32 / FP16 / INT8 evaluation on the 20 calib clips

**Purpose.** Build TensorRT FP32, FP16 and INT8 engines for YOLO11s at 640 px. INT8 is
calibrated on the 500 Step 1 calib frames only. Run each engine, plus the PyTorch FP32
reference, on the **20 calib clips (~3.95k frames)**, cache every detection, and compute
pycocotools mAP@0.5 and mAP@0.5:0.95 overall and per slice. Then run the sanity checks:
- FP16 ≈ FP32.
- INT8 slightly lower.
- TensorRT FP32 ≈ PyTorch FP32.
- Our pipeline ≈ Ultralytics `model.val()` (PLAN §5.3).
- The microbenchmark and the pre-registered model-size rule.

**The test split is not touched.** Calib/test clip lists and slice labels come from the
committed `results/bdd100k_mot/bdd_subsets/`. The calib ground truth is rebuilt from the labels
and must match Step 1's counts exactly, or the notebook stops.

**Before running**
1. *Add Data* → **`tcjpaperdatasets`** (same as notebook 01). No other inputs; the YOLO11s
   weights are downloaded by Ultralytics (Internet On) and their SHA-256 is recorded.
2. *Settings* → **Accelerator: GPU T4 x2** (one GPU pinned), **Internet: On**.
3. *Add-ons → Secrets* → **`GH_TOKEN`** attached.
4. First run with `SMOKE_TEST = True` (2 clips, 50 INT8 calibration frames; ≈15 min, mostly
   engine builds). Then `SMOKE_TEST = False` → *Save Version → Save & Run All* (≈25–30 min).
   If the model-size rule triggers (cell 5 says so), set
   `CONFIG["model_override"] = {"name": "yolo11m", "weights": "yolo11m.pt"}` and run again.

**Download afterwards:** `results_02_eval_fp32_fp16_int8.zip` → unzip into the repo root
(`results/bdd100k_mot/calib_*`, `results/microbench/`, `results/sanity/`, `logs/`).
**Keep this notebook version**: its output `work02/cache/` (per-frame detections) is the input
of notebook 03 (*Add Data → Notebook Output*). The final cell raises if any sanity check failed,
after the zip is written, so the failure is visible and nothing is lost.


In [ ]:
# ===================== CELL 1: CONFIG + ENVIRONMENT CHECK =====================
# Edit nothing below for a normal run except SMOKE_TEST.
SMOKE_TEST = True   # True: tiny subset (run this first). False: full run.
NOTEBOOK_REV = "d6c843f60e48"   # hash of this notebook's generated source; checked against the cloned repo

CONFIG = {
    "notebook": "02_eval_fp32_fp16_int8",
    "repo_https": "https://github.com/vinaykamble-nsio/adapres.git",       # private repo, cloned with Secret GH_TOKEN
    "repo_ref": "main",
    "repo_dir": "/tmp/adapres",                     # /tmp: not saved into notebook output
    "repo_zip_marker": "adapres/GIT_COMMIT",        # fallback: repo zip attached as a dataset
    "config_path": "configs/default.yaml",          # relative to repo_dir
    "overrides": {                                  # applied on top of default.yaml
        "paths": {"input_root": "/kaggle/input", "work_root": "/kaggle/working"},
    },
    "cuda_visible_devices": "0",                    # T4 x2 -> pin to one GPU
    "required_gpu_substring": "T4",
    "model_override": None,   # e.g. {"name": "yolo11m", "weights": "yolo11m.pt"} if the size rule triggers
}

import os, sys, subprocess, json, platform, base64, glob, shutil, time
os.environ["CUDA_VISIBLE_DEVICES"] = CONFIG["cuda_visible_devices"]  # before any CUDA import
T0 = time.time()

def sh(cmd, check=True):
    """Run a shell command, print its output, raise with the real error on failure."""
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.stderr.strip():
        print(r.stderr[-6000:], file=sys.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

sh("nvidia-smi")
print("python", sys.version)
for mod in ("torch", "onnxruntime", "tensorrt", "ultralytics"):
    try:
        m = __import__(mod)
        print(f"{mod} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{mod} NOT AVAILABLE YET ({e!r}); installed in cell 2 if pinned")
import torch
print("torch.cuda.is_available:", torch.cuda.is_available(), "| visible devices:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
gpu = torch.cuda.get_device_name(0)
print("GPU 0:", gpu)
if CONFIG["required_gpu_substring"] not in gpu:
    raise RuntimeError(f"GPU is {gpu!r}; this project requires a T4 (P100 has no fast INT8). "
                       "Settings -> Accelerator -> GPU T4 x2, then restart.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"expected exactly 1 visible GPU after pinning, got {torch.cuda.device_count()}")

# ---- get the repo: private GitHub clone (Secret GH_TOKEN), else attached zip dataset
REPO = CONFIG["repo_dir"]
repo_source = None
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
try:
    if "github.com/OWNER/" in CONFIG["repo_https"]:
        raise RuntimeError("repo_https is still the placeholder: set it in configs/kaggle.yaml and regenerate")
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    if not token:
        raise RuntimeError("Secret GH_TOKEN is empty")
    hdr = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    # token passed as a one-off header: never written to .git/config or printed
    r = subprocess.run(["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {hdr}",
                        "clone", "--quiet", CONFIG["repo_https"], REPO], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed: " + r.stderr.replace(token, "***").replace(hdr, "***"))
    sh(["git", "-C", REPO, "checkout", "--quiet", CONFIG["repo_ref"]])
    repo_source = "github"
    del token, hdr
except Exception as e:
    print(f"GitHub clone unavailable: {e!r}\nTrying zip-dataset fallback ...")
    hits = sorted(glob.glob("/kaggle/input/**/" + CONFIG["repo_zip_marker"], recursive=True))
    if not hits:
        raise RuntimeError("Neither GitHub clone (Secret GH_TOKEN) nor an attached repo zip "
                           "(dataset containing adapres/GIT_COMMIT) is available. See README cell.") from e
    shutil.copytree(os.path.dirname(hits[0]), REPO)
    repo_source = "zip_dataset:" + os.path.dirname(hits[0])

sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, os.path.join(REPO, "scripts"))
# Re-running cells in a live kernel re-clones the repo but Python would keep the OLD imported
# modules while GIT records the NEW commit. Drop them so the code always matches the hash.
_stale = [m for m in list(sys.modules) if m == "adapres" or m.startswith("adapres.") or m == "make_notebooks"]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"dropped {len(_stale)} previously imported adapres modules (fresh code from this clone)")
import make_notebooks
expected_rev = make_notebooks.notebook_rev(CONFIG["notebook"] + ".ipynb")
if expected_rev != NOTEBOOK_REV:
    raise RuntimeError(f"STALE NOTEBOOK: this notebook is rev {NOTEBOOK_REV} but repo {CONFIG['repo_ref']} "
                       f"generates rev {expected_rev}. Re-import notebooks/{CONFIG['notebook']}.ipynb "
                       "from the repo (File -> Import Notebook), or push your local commits.")
print("notebook rev", NOTEBOOK_REV, "matches repo")
from adapres import config as C
from adapres.repro import git_commit, set_seed
cfg = C.load(os.path.join(REPO, CONFIG["config_path"]), CONFIG["overrides"])
SEED = cfg["seed"]
set_seed(SEED)
GIT = git_commit(REPO)
print("repo source:", repo_source)
print("git commit:", GIT)
print("seed:", SEED, "| SMOKE_TEST:", SMOKE_TEST)
if GIT["commit"] is None:
    raise RuntimeError("could not determine the git commit; refusing to produce unattributable results")
if GIT["dirty"]:
    raise RuntimeError("repo checkout is dirty; results would not match the recorded commit")
RESULTS = C.resolve_out(cfg, "results_dir", SMOKE_TEST)
LOGS = C.resolve_out(cfg, "logs_dir", SMOKE_TEST)
for d in (RESULTS, LOGS):
    os.makedirs(d, exist_ok=True)
print("results ->", RESULTS, "| logs ->", LOGS)


In [ ]:
# ===================== CELL 2: LOCKED INSTALLS + VERSION CHECK =====================
# Internet must be ON. Installs only what the Kaggle image lacks; torch is not touched.
# Every pin is exact (locked 2026-11-20); the cell fails if anything installs at another version.
import re, importlib.metadata as md
print("python", sys.version, "| torch", torch.__version__, "| torch CUDA", torch.version.cuda)
REQ = os.path.join(REPO, "requirements-kaggle.txt")
lines = [ln.split("#")[0].strip() for ln in open(REQ)]
opts = [tok for ln in lines if ln.startswith("-") for tok in ln.split()]  # e.g. --extra-index-url URL
print("pip options:", opts)
reqs = [ln for ln in lines if ln and not ln.startswith("-")]
PIP = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check"]
# Preflight: resolve each requirement on its own (no install) so ALL failures show in one run.
bad = {}
for req in reqs:
    p = subprocess.run(PIP + ["--dry-run", "--no-deps", "--quiet", *opts, req], capture_output=True, text=True)
    if p.returncode != 0:
        bad[req] = "\n".join(l for l in p.stderr.splitlines() if "ERROR" in l) or p.stderr[-2000:]
    print(f"preflight {req:45s} {'OK' if p.returncode == 0 else 'FAILED'}")
if bad:
    for req, err in bad.items():
        print(f"\n--- {req} ---\n{err}")
    raise RuntimeError(f"{len(bad)} requirement(s) have no wheel for this Python/CUDA: {list(bad)}. "
                       "Paste this whole cell output; the available versions are listed above.")
r = sh(PIP + ["--quiet", "-r", REQ], check=False)
with open(os.path.join(LOGS, f"{CONFIG['notebook']}_pip.log"), "w") as f:
    f.write(r.stdout + "\n--- stderr ---\n" + r.stderr)
if r.returncode != 0:
    raise RuntimeError("pip install failed (real error above).")
importlib.invalidate_caches()
# Version lock: record exactly what got installed so requirements-kaggle.txt can be pinned to it.
names = [re.split(r"[<>=!~\[ ]", q)[0] for q in reqs]
resolved = {n: md.version(n) for n in names}
os.makedirs(os.path.join(RESULTS, "env"), exist_ok=True)
lock = "\n".join(f"{n}=={v}" for n, v in resolved.items())
with open(os.path.join(RESULTS, "env", "pip_resolved.txt"), "w") as f:
    f.write(f"# python {sys.version.split()[0]} torch {torch.__version__} cuda {torch.version.cuda}\n{lock}\n")
print("RESOLVED VERSIONS (copy into requirements-kaggle.txt as exact pins):\n" + lock)
trt_dists = ("tensorrt-cu13", "tensorrt-cu13-bindings", "tensorrt-cu13-libs")
if len({resolved.get(d) for d in trt_dists}) != 1:
    raise RuntimeError(f"TensorRT wrapper/bindings/libs version mismatch: {resolved}")
pinned = {re.split(r"[<>=!~\[ ]", q)[0]: q.split("==")[-1] for q in reqs if "==" in q}
drift = {n: (pinned[n], resolved[n]) for n in pinned if pinned[n] != resolved[n]}
if drift:
    raise RuntimeError(f"installed versions differ from the lock (pinned, installed): {drift}")
try:
    import tensorrt
except ImportError:
    # Diagnose instead of guessing: which top-level modules did each TensorRT dist install?
    for d in trt_dists:
        try:
            tops = sorted({str(f).split("/")[0] for f in (md.files(d) or [])})
            print(f"{d}: top-level entries {tops[:25]}")
        except md.PackageNotFoundError:
            print(f"{d}: NOT INSTALLED")
    raise
import onnx, onnxruntime
print("import OK: tensorrt", tensorrt.__version__, "| onnx", onnx.__version__, "| onnxruntime", onnxruntime.__version__)
print("torch still", torch.__version__, "cuda", torch.version.cuda)


In [ ]:
# ===================== CELL 3: PREPARE (manifest, GT rebuild, weights) =====================
import logging
os.environ["YOLO_AUTOINSTALL"] = "False"   # never let Ultralytics pip-install unpinned packages
if CONFIG["model_override"]:
    cfg = C.load(os.path.join(REPO, CONFIG["config_path"]), {**CONFIG["overrides"], "model": CONFIG["model_override"]})
logging.basicConfig(level=logging.INFO, force=True, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(os.path.join(LOGS, "02_eval.log"))])
from adapres.pipeline.eval_fixed import Step2
WORK = os.path.join(cfg["paths"]["work_root"], "smoke" if SMOKE_TEST else "", "work02")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)                              # Ultralytics downloads weights into the cwd
from ultralytics import YOLO
import ultralytics
print("ultralytics", ultralytics.__version__)
_m = YOLO(cfg["model"]["weights"])
WEIGHTS = os.path.abspath(cfg["model"]["weights"])
if not os.path.exists(WEIGHTS):
    raise RuntimeError(f"weights not found after download: {WEIGHTS}")
NAMES = dict(_m.names)
del _m
step = Step2(cfg, SMOKE_TEST, REPO)
step.coco_names = NAMES
t = time.time()
info = step.prepare()
print(json.dumps(info, indent=2)[:2000])
print(f"prepare: {time.time()-t:.0f}s | model {cfg['model']['name']} weights {WEIGHTS}")


In [ ]:
# ===================== CELL 4: EXPORT TensorRT ENGINES (rebuilt in this session) =====================
t = time.time()
int8_yaml = step.int8_dataset(NAMES)
print("INT8 calibration frames (calib split only):", step.n_int8_frames)
eng = step.export(WEIGHTS, int8_yaml)
for lv, e in eng.items():
    md = e.get("ultralytics_metadata", {})
    print(f"{lv:5s} {e['bytes']/1e6:7.1f} MB build {e['build_s']:6.1f}s sha {e['sha256'][:12]} "
          f"| meta: half={md.get('half')} int8={md.get('int8')} dynamic={md.get('dynamic')} batch={md.get('batch')} "
          f"imgsz={md.get('imgsz')} {md.get('_error', '')}")
print(f"export: {(time.time()-t)/60:.1f} min")


In [ ]:
# ===================== CELL 5: MICROBENCH + MODEL-SIZE RULE (PLAN §1.2) =====================
t = time.time()
BACKENDS = step.backends()
mb = step.microbench(BACKENDS)
for name, r in mb["per_backend"].items():
    print(f"{name:8s} infer median {r['infer_ms']['median']:6.2f} ms  p95 {r['infer_ms']['p95']:6.2f}  "
          f"| wall median {r['wall_ms']['median']:6.2f} ms")
rule = mb.get("size_rule")
print("size rule:", rule)
if rule and rule["triggered"]:
    print("\n" + "!" * 80 + "\nMODEL-SIZE RULE TRIGGERED: INT8 is less than "
          f"{100*rule['threshold']:.0f}% faster than FP16. Per PLAN §1.2, ALSO run this notebook with\n"
          'CONFIG["model_override"] = {"name": "yolo11m", "weights": "yolo11m.pt"}\n' + "!" * 80)
print(f"microbench: {time.time()-t:.0f}s")


In [ ]:
# ===================== CELL 6: CACHE RUNS (all backends x all calib frames; resumable) =====================
t = time.time()
done = step.cache_runs(BACKENDS)
print("cached sequences per backend:", done, f"| {(time.time()-t)/60:.1f} min")


In [ ]:
# ===================== CELL 7: EVALUATE (pycocotools, per slice) + dev oracle =====================
import pandas as pd
t = time.time()
METRICS = step.evaluate()
rows = []
for name, m in METRICS.items():
    r = {"policy": name, "mAP50-95": 100*m["all"]["map50_95"], "mAP50": 100*m["all"]["map50"],
         "meanF1@.5": m["all"]["mean_f1_05"]}
    for s in ("easy", "crowded", "night", "scene_change_natural"):
        r[f"{s} mAP50-95"] = 100*m[s]["map50_95"] if m[s]["n_images"] else None
    tc = m["all"].get("timing_cache")
    if tc:
        r["infer ms (med)"] = tc["infer_ms"]["median"]; r["wall ms (med)"] = tc["wall_ms"]["median"]
    rows.append(r)
pd.set_option("display.width", 250)
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("slice sizes:", {k: v["n_images"] for k, v in METRICS["fp16"].items()})
print("NOTE: timings are single-pass cache timings (JPEG decode included), not the Step 4 latency protocol.")
print(f"evaluate: {time.time()-t:.0f}s")


In [ ]:
# ===================== CELL 8: PIPELINE vs ULTRALYTICS model.val() (PLAN §5.3) =====================
t = time.time()
vc = step.val_check()
print(json.dumps(vc, indent=2))
print(f"val check: {time.time()-t:.0f}s")


In [ ]:
# ===================== CELL 9: SANITY CHECKS =====================
SAN = step.sanity(METRICS)
for c in SAN["checks"]:
    tag = ("PASS" if c["pass"] else "FAIL") if c.get("binding", True) else ("info" if c["pass"] else "INFO-FAIL")
    print(f"[{tag}] {c['check']:58s} value={c['value_pts']} pts  rule: {c['rule']}")
print("ALL SANITY CHECKS PASS" if SAN["all_pass"] else "\n*** SANITY FAILURES ABOVE: results are saved; do not proceed to Step 3 until reviewed ***")


In [ ]:
# ===================== FINAL CELL: ZIP RESULTS =====================
import zipfile
suffix = "_smoke" if SMOKE_TEST else ""
zpath = f"/kaggle/working/results_02_eval_fp32_fp16_int8{suffix}.zip"
base = os.path.dirname(RESULTS)  # contains results/ and logs/
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for top in (RESULTS, LOGS):
        for root, _, files in os.walk(top):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, base))
                n += 1
print(f"wrote {zpath} ({n} files, {os.path.getsize(zpath)/1e6:.2f} MB)")
print(f"total notebook time: {(time.time()-T0)/60:.1f} min")
if not SAN["all_pass"]:
    raise RuntimeError("sanity check(s) failed (zip written): " + str([c["check"] for c in SAN["checks"] if not c["pass"] and c.get("binding", True)]))
